<a href="https://colab.research.google.com/github/sandhya-sandy09/Multi-LLM-Scoring-Cognitive-assessment/blob/main/cognitive_test_code_students.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[groq api key link](https://console.groq.com/home)

[openrouter api key link](https://openrouter.ai/)

In [ ]:
!pip install -q requests==2.32.5

In [ ]:
!pip install -q langchain langchain-community langchain-openai langchain-groq pypdf faiss-cpu sentence-transformers fpdf

In [ ]:
!pip install -q langchain langchain-community langchain-openai langchain-groq pypdf faiss-cpu sentence-transformers fpdf

import os
import time
import json
import pandas as pd
import re
from typing import List, Dict, Any
from google.colab import files
from fpdf import FPDF

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.exceptions import OutputParserException

def get_config():
    print("=" * 60)
    print("API KEY CONFIGURATION")
    print("=" * 60)

    config = {}

    config["GROQ_API_KEY"] = input("Enter GROQ API KEY: ").strip()
    config["OPENROUTER_API_KEY"] = input("Enter OPENROUTER API KEY: ").strip()

    config["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"

    config["GEN_MODEL"] = "llama-3.3-70b-versatile"
    config["MAP_MODEL"] = "llama-3.1-8b-instant"

    config["EVAL_MODELS"] = [
        "liquid/lfm-2.5-1.2b-thinking:free",
        "nvidia/nemotron-3-nano-30b-a3b:free",
        "moonshotai/kimi-k2:free"
    ]

    return config

def setup_rag():
    print("\n" + "="*60)
    print("PDF UPLOAD & PROCESSING")
    print("="*60)

    uploaded = files.upload()
    if not uploaded:
        print("No file uploaded.")
        return None

    pdf_path = list(uploaded.keys())[0]
    print(f"Processing {pdf_path}...")

    loader = PyPDFLoader(pdf_path)
    docs = loader.load()

    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    splits = splitter.split_documents(docs)

    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vectorstore = FAISS.from_documents(splits, embeddings)

    return vectorstore.as_retriever(search_kwargs={"k": 4})

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def invoke_with_retry(chain, input_data, max_retries=3):
    for attempt in range(max_retries):
        try:
            return chain.invoke(input_data)
        except Exception as e:
            if "429" in str(e) or "Rate limit" in str(e):
                wait_time = (attempt + 1) * 5
                print(f"Rate limited. Retrying in {wait_time}s...")
                time.sleep(wait_time)
            elif attempt == max_retries - 1:
                raise e
            else:
                print(f"Error: {e}. Retrying...")
                time.sleep(2)

def generate_questions(topic, retriever, llm):
    print(f"\nGenerating questions for: {topic}")

    parser = JsonOutputParser()

    template = """You are an expert exam creator. create exactly 36 multiple-choice questions based EXCLUSIVELY on the provided context.

    REQUIREMENTS:
    1. Generate exactly 36 questions.
    2. Mix Bloom's Taxonomy levels:
       - 6 Remember (K1)
       - 6 Understand (K2)
       - 6 Apply (K3)
       - 6 Analyze (K4)
       - 6 Evaluate (K5)
       - 6 Create (K6)
    3. questions from all level should be there .
    3. Output must be a valid JSON object with a single key "questions" containing a list of objects.

    JSON STRUCTURE PER QUESTION:
    {{
        "id": 1,
        "text": "Question text here?",
        "options": {{
            "A": "Option A text",
            "B": "Option B text",
            "C": "Option C text",
            "D": "Option D text"
        }},
        "correct_answer": "A",
        "blooms_level": "K1"
    }}

    CONTEXT:
    {context}

    TOPIC: {topic}

    {format_instructions}
    """

    prompt = ChatPromptTemplate.from_template(template)

    chain = (
        {"context": retriever | format_docs, "topic": RunnablePassthrough()}
        | prompt.partial(format_instructions=parser.get_format_instructions())
        | llm
        | parser
    )

    try:
        response = invoke_with_retry(chain, topic)
        questions = response.get("questions", [])

        if len(questions) < 36:
            print(f"Warning: Only generated {len(questions)} questions.")

        return questions[:36]

    except Exception as e:
        print(f"Generation Error: {e}")
        return []

def conduct_test(questions):
    user_data = []
    print("\n" + "="*60)
    print("STARTING ASSESSMENT")
    print("="*60)

    input("Press Enter to begin...")

    for q in questions:
        print(f"\nQ{q['id']} [{q.get('blooms_level', 'General')}]: {q['text']}")
        for key, val in q['options'].items():
            print(f"{key}. {val}")

        start_time = time.time()
        choice = input("Answer (A/B/C/D): ").strip().upper()
        duration = time.time() - start_time

        if choice not in ['A', 'B', 'C', 'D']:
            choice = 'A'

        is_correct = (choice == q['correct_answer'])

        user_data.append({
            "id": q['id'],
            "level": q.get('blooms_level', 'K1'),
            "correct": is_correct,
            "time": duration,
            "user_ans": choice,
            "correct_ans": q['correct_answer']
        })

    return pd.DataFrame(user_data)

def get_ai_analysis(df, topic, config):
    print("\n" + "="*60)
    print("GENERATING AI ANALYSIS (MULTI-MODEL)")
    print("="*60)

    stats = df.groupby("level").agg({
        "correct": ["mean", "count"],
        "time": "mean"
    }).round(2)

    overall_score = df["correct"].mean() * 100

    context_str = f"""
    Topic: {topic}
    Overall Score: {overall_score:.1f}%

    Performance by Bloom's Level (K1–K6):
    {stats.to_string()}

    Detailed Question-Level Data:
    {df.to_string()}
    """

    template = """You are an expert educational assessment analyst.

Analyze the student's performance across Bloom's Taxonomy levels.

DATA:
{data}

Provide:
1. Overall Performance Summary
2. Bloom’s Taxonomy Level-wise Analysis (K1–K6)
3. Time Management Analysis
4. OVERALL PERFORMANCE SCORE OUT OF 10
5. Cognitive Strengths & Weaknesses
6. Actionable Study Recommendations
"""

    prompt = ChatPromptTemplate.from_template(template)
    analysis_results = {}
    successful_models = 0

    for model in config["EVAL_MODELS"]:
        print(f"Running evaluation using: {model}...")

        try:
            llm = ChatOpenAI(
                model=model,
                api_key=config["OPENROUTER_API_KEY"],
                base_url=config["OPENAI_API_BASE"],
                temperature=0.3
            )

            chain = prompt | llm
            analysis = invoke_with_retry(chain, {"data": context_str})
            analysis_results[model] = analysis.content
            successful_models += 1

        except Exception as e:
            print(f"Model {model} failed: {e}")
            continue

    if successful_models == 0:
        print("\nAll evaluation models failed.")
    else:
        print(f"\nEvaluation completed using {successful_models} model(s).")

    return analysis_results, stats

def clean_text(text):
    return str(text).encode('latin-1', 'replace').decode('latin-1')

def generate_pdf_report(topic, questions, results_df, analyses, stats):
    print("\nGenerating PDF Report...")
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()

    pdf.set_font("Arial", "B", 16)
    pdf.cell(0, 10, clean_text(f"Assessment Report: {topic}"), ln=True, align="C")
    pdf.ln(10)

    pdf.set_font("Arial", "B", 14)
    pdf.cell(0, 10, "1. Statistical Summary", ln=True)
    pdf.set_font("Arial", "", 12)

    total_score = results_df['correct'].sum()
    total_q = len(results_df)
    accuracy = (total_score / total_q) * 100

    pdf.cell(0, 10, f"Total Score: {total_score}/{total_q}", ln=True)
    pdf.cell(0, 10, f"Accuracy: {accuracy:.1f}%", ln=True)
    pdf.ln(5)

    pdf.set_font("Courier", "", 10)
    stats_str = stats.to_string()
    for line in stats_str.split('\n'):
        pdf.cell(0, 5, clean_text(line), ln=True)

    pdf.add_page()
    pdf.set_font("Arial", "B", 14)
    pdf.cell(0, 10, "2. Detailed Question Review", ln=True)
    pdf.ln(5)

    q_map = {q['id']: q for q in questions}

    for idx, row in results_df.iterrows():
        qid = row['id']
        q_data = q_map.get(qid, {})
        user_ans = row['user_ans']
        corr_ans = row['correct_ans']
        is_corr = row['correct']

        pdf.set_font("Arial", "B", 11)
        pdf.multi_cell(0, 6, clean_text(f"Q{qid} [{row['level']}]: {q_data.get('text', 'N/A')}"))

        pdf.set_font("Arial", "", 10)
        options = q_data.get('options', {})
        for opt_key, opt_val in options.items():
            prefix = "  "
            if opt_key == user_ans: prefix += "(YOUR ANSWER) "
            if opt_key == corr_ans: prefix += "(CORRECT) "

            pdf.multi_cell(0, 5, clean_text(f"{prefix}{opt_key}: {opt_val}"))

        status = "CORRECT" if is_corr else "INCORRECT"
        pdf.set_font("Arial", "I", 10)
        pdf.cell(0, 6, f"Result: {status} | Time: {row['time']:.2f}s", ln=True)
        pdf.ln(5)

    pdf.add_page()
    pdf.set_font("Arial", "B", 14)
    pdf.cell(0, 10, "3. AI Analysis Reports", ln=True)

    for model_name, report_text in analyses.items():
        pdf.ln(5)
        pdf.set_font("Arial", "B", 12)
        pdf.set_fill_color(230, 230, 230)
        pdf.cell(0, 10, clean_text(f"Analysis by {model_name}"), ln=True, fill=True)
        pdf.ln(2)

        pdf.set_font("Arial", "", 10)
        pdf.multi_cell(0, 5, clean_text(report_text))
        pdf.ln(10)

    filename = "Assessment_Report_Full.pdf"
    pdf.output(filename)
    return filename

def main():
    config = get_config()

    retriever = setup_rag()
    if not retriever:
        return

    topic = input("\nEnter assessment topic: ").strip()

    llm_gen = ChatGroq(
        model=config["GEN_MODEL"],
        api_key=config["GROQ_API_KEY"],
        temperature=0.3
    )

    questions = generate_questions(topic, retriever, llm_gen)

    if not questions:
        print("Failed to generate questions. Exiting.")
        return

    print(f"\nSuccessfully generated {len(questions)} questions.")

    results_df = conduct_test(questions)

    print("\n" + "="*60)
    print("STATISTICS")
    print("="*60)
    print(f"Total Score: {results_df['correct'].sum()}/{len(results_df)}")
    print(f"Accuracy: {(results_df['correct'].mean()*100):.1f}%")

    analyses, stats_summary = get_ai_analysis(results_df, topic, config)

    print("\n" + "="*60)
    print("DISPLAYING ANALYSIS")
    print("="*60)
    for model, text in analyses.items():
        print(f"\n--- {model} ---")
        print(text[:500] + "...\n(Full report in PDF)")

    pdf_filename = generate_pdf_report(topic, questions, results_df, analyses, stats_summary)

    print(f"\nPDF generated: {pdf_filename}")
    files.download(pdf_filename)

if __name__ == "__main__":
    main()

[Response form link](https://forms.gle/NFF3suJ1iLqJggPe8)